<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Launch Sites Locations Analysis with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:
- **TASK 1:** Mark all launch sites on a map
- **TASK 2:** Mark the success/failed launches for each site on the map
- **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [1]:
import folium
import wget
import pandas as pd

In [2]:
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

from utils.paths import DATA_DIR
from src.map_folium import assign_marker_color
from src.map_folium import calculate_distance

## Task 1: Mark all launch sites on a map


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site. 


In [3]:
spacex_df = pd.read_csv(DATA_DIR/'spacex_launch_geo.csv')

Now, you can take a look at what are the coordinates for each site.


In [4]:
# Select relevant sub-columns: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df.head(5)

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [6]:
# Create a blue circle at NASA Johnson Space Center's coordinate with a popup label showing its name
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Create a blue circle at NASA Johnson Space Center's coordinate with a icon showing its name
marker = folium.map.Marker(
    nasa_coordinate,
    # Create an icon as a text label
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

In [7]:
coor_CCAFS_LC_40 = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'CCAFS LC-40',
    ['Lat', 'Long']
].values.tolist()[0]

coor_CCAFS_SLC_40 = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'CCAFS SLC-40',
    ['Lat', 'Long']
].values.tolist()[0]

coor_KSC_LC_39A = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'KSC LC-39A',
    ['Lat', 'Long']
].values.tolist()[0]

coor_VAFB_SLC_4E = launch_sites_df.loc[
    launch_sites_df['Launch Site'] == 'VAFB SLC-4E',
    ['Lat', 'Long']
].values.tolist()[0]


# Initialize map
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)


# Circles
circle_1 = folium.Circle(
    location=coor_CCAFS_LC_40,
    radius=1000,
    color='#d35400',
    fill=True,
    popup='CCAFS LC-40'
)

circle_2 = folium.Circle(
    location=coor_CCAFS_SLC_40,
    radius=1000,
    color='#2980b9',
    fill=True,
    popup='CCAFS SLC-40'
)

circle_3 = folium.Circle(
    location=coor_KSC_LC_39A,
    radius=1000,
    color='#27ae60',
    fill=True,
    popup='KSC LC-39A'
)

circle_4 = folium.Circle(
    location=coor_VAFB_SLC_4E,
    radius=1000,
    color='#8e44ad',
    fill=True,
    popup='VAFB SLC-4E'
)


# Markers
marker_1 = folium.Marker(
    location=coor_CCAFS_LC_40,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#d35400;"><b>CCAFS LC-40</b></div>'
    )
)

marker_2 = folium.Marker(
    location=coor_CCAFS_SLC_40,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#2980b9;"><b>CCAFS SLC-40</b></div>'
    )
)

marker_3 = folium.Marker(
    location=coor_KSC_LC_39A,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#27ae60;"><b>KSC LC-39A</b></div>'
    )
)

marker_4 = folium.Marker(
    location=coor_VAFB_SLC_4E,
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12px; color:#8e44ad;"><b>VAFB SLC-4E</b></div>'
    )
)


# Add to map
circle_1.add_to(site_map)
marker_1.add_to(site_map)

circle_2.add_to(site_map)
marker_2.add_to(site_map)

circle_3.add_to(site_map)
marker_3.add_to(site_map)

circle_4.add_to(site_map)
marker_4.add_to(site_map)

site_map

The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:
- Are all launch sites in proximity to the Equator line?
No, the Launch sites are above the equator line
- Are all launch sites in very close proximity to the coast?
No, one of them is not close to the coast

# Task 2: Mark the success/failed launches for each site on the map


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


Next, let's create markers for all launch records. 
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


In [8]:
marker_cluster = MarkerCluster()

_TODO:_ Create a new column in `launch_sites` dataframe called `marker_color` to store the marker colors based on the `class` value


In [9]:
spacex_df['marker_color'] = spacex_df['class'].map({1: 'green', 0:'red'})

In [10]:
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [11]:
# Add marker_cluster to current site_map
site_map.add_child(marker_cluster)

# for each row in spacex_df data frame
for index, record in spacex_df.iterrows():

    # create Marker object
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(
            color='white',
            icon_color=record['marker_color'],
            icon='info-sign'
        )
    )

    # add marker to cluster
    marker_cluster.add_child(marker)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


# TASK 3: Calculate the distances between a launch site to its proximities


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [12]:
# Add Mouse Position to get the coordinate (Lat, Long) for a mouse over on the map
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


You can calculate the distance between two points on the map based on their `Lat` and `Long` values using the following method:


_TODO:_ Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [17]:
# find coordinate of the closet coastline
launch_site_lat = 34.63293
launch_site_lon = -120.611
coastline_lat = 28.56155
coastline_lon = -80.56755

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

_TODO:_ After obtained its coordinate, create a `folium.Marker` to show the distance


In [18]:
# folium.Marker to display the distance between coastline point and launch site using the icon property 
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_coastline),
    )
)



_TODO:_ Draw a `PolyLine` between a launch site to the selected coastline point


In [19]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)
site_map.add_child(distance_marker)
site_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [20]:
railway_lat = 28.57435
railway_lon = -80.80462

highway_lat = 28.56259
highway_lon = -80.57068

city_lat = 28.47593
city_lon = -81.05164

# Distances

distance_city = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    city_lat,
    city_lon
)

distance_rail = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    railway_lat,
    railway_lon
)

distance_high = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    highway_lat,
    highway_lon
)

# Markers 

city_marker = folium.Marker(
    [city_lat, city_lon],

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_city),
    )
)

rail_marker = folium.Marker(
    [railway_lat, railway_lon],

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_rail),
    )
)  

high_marker = folium.Marker(
    [highway_lat, highway_lon],

    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),

        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>'
        % "{:10.2f} KM".format(distance_high),
    )
)  

## Coordinates
coordinates_city = [
    [launch_site_lat, launch_site_lon],
    [city_lat, city_lon]
]

coordinates_rail = [
    [launch_site_lat, launch_site_lon],
    [railway_lat, railway_lon]
]

coordinates_high = [
    [launch_site_lat, launch_site_lon],
    [highway_lat, highway_lon]
]

#Lines
line_city = folium.PolyLine(
    locations=coordinates_city,
    weight=1
) 

line_rail = folium.PolyLine(
    locations=coordinates_rail,
    weight=1
) 

line_high = folium.PolyLine(
    locations=coordinates_high,
    weight=1
) 

# Site map
site_map.add_child(city_marker)
site_map.add_child(rail_marker)
site_map.add_child(high_marker)

site_map.add_child(line_city)
site_map.add_child(line_rail)
site_map.add_child(line_high)
site_map

# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


Joseph Santarcangelo


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2021-05-26|1.0|Yan|Created the initial version|


Copyright © 2021 IBM Corporation. All rights reserved.
